In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

/home/hanxd/Repositories/ESP/our_codes


In [ ]:
anotherdata = pd.read_csv(our_data + "ginnascores.txt", sep="\t", header=0)
result = anotherdata["complex_name"].str.split(r"_", expand=True)
result.columns = ["enzyme", "substrate", "cc"]
result["substrate_cc"] = result["substrate"] + result["cc"]
anotherdata = pd.concat([anotherdata, result], axis=1)

In [3]:
anotherdata

,complex_name,wei_score,if_right,enzyme,substrate,cc,substrate_cc
0,CYP707A1_08Y_C22,0.345,1,CYP707A1,08Y,C22,08YC22
1,CYP76B10_08Y_C22,0.387,1,CYP76B10,08Y,C22,08YC22
2,CYP72A208_08Y_C22,0.369,1,CYP72A208,08Y,C22,08YC22
3,CYP725A4_08Y_C22,0.341,1,CYP725A4,08Y,C22,08YC22
4,CYP71AY5_08Y_C22,0.395,1,CYP71AY5,08Y,C22,08YC22
...,...,...,...,...,...,...,...
75056,CYP72A610_AHT_C8,0.300,1,CYP72A610,AHT,C8,AHTC8
75057,CYP74L1_AHT_C8,0.249,1,CYP74L1,AHT,C8,AHTC8
75058,CYP79A12_AHT_C8,0.248,1,CYP79A12,AHT,C8,AHTC8
75059,CYP71D12_AHT_C8,0.229,1,CYP71D12,AHT,C8,AHTC8


In [ ]:
anotherdata_N = anotherdata.sort_values(
    by=["substrate_cc", "wei_score"], ascending=[True, False]
)
anotherdata_N["ranking"] = anotherdata_N.groupby("substrate_cc").cumcount() + 1

In [5]:
anotherdata_N

,complex_name,wei_score,if_right,enzyme,substrate,cc,substrate_cc,ranking
4,CYP71AY5_08Y_C22,0.395,1,CYP71AY5,08Y,C22,08YC22,1
1,CYP76B10_08Y_C22,0.387,1,CYP76B10,08Y,C22,08YC22,2
9,CYP72A610_08Y_C22,0.386,1,CYP72A610,08Y,C22,08YC22,3
21,CYP92B14_08Y_C22,0.386,1,CYP92B14,08Y,C22,08YC22,4
8,CYP71D184_08Y_C22,0.381,1,CYP71D184,08Y,C22,08YC22,5
...,...,...,...,...,...,...,...,...
72937,CYP87D20_ZMP_C18,0.080,1,CYP87D20,ZMP,C18,ZMPC18,194
72947,CYP98A112_ZMP_C18,0.072,1,CYP98A112,ZMP,C18,ZMPC18,195
72946,CYP72A613_ZMP_C18,0.071,1,CYP72A613,ZMP,C18,ZMPC18,196
72944,2X5W_ZMP_C18,0.069,1,2X5W,ZMP,C18,ZMPC18,197


In [ ]:
anotherdata_N.to_csv(our_data + "paper/gninaRanking.csv")

In [ ]:
anotherdata_Nt = anotherdata_N[anotherdata_N["if_right"] == 2]

In [ ]:
anotherdata_Nt.groupby(["substrate"]).size().unique()

array([ 1,  2, 11,  3,  4,  5])

In [ ]:
anotherdata_Nt.groupby(["enzyme"]).size().unique()

array([1])

In [ ]:
anotherdata_Nt.groupby(["enzyme", "substrate"]).size().unique()

array([1])

In [ ]:
print(anotherdata_N["enzyme"].nunique())
print(anotherdata_N["substrate"].nunique())
print(anotherdata_N["substrate_cc"].nunique())
print(294 * 252)
print(294 * 311)

294
252
311
74088
91434
